# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2 Mass Spectrometry Human Mast Cell](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. First, we inspect the package and record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset:", metadata.name)
print("Description:", metadata.description)
if hasattr(metadata, 'keywords'):
    print("Keywords: ", metadata.keywords)

## 2. Data Overview
Let's enumerate the record sets and explore the available fields and IDs.

We'll print all record sets with their `@id`, names, and their fields/columns with their corresponding `@id`s for reference.

In [ ]:
# List all record sets and their fields with @id for reference
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in this schema. Check if there are files with tabular data.")
else:
    for rs in dataset.record_sets:
        print(f"Record Set: {rs['@id']}")
        print(f"  Name: {rs.get('name', '<no name>')}")
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                if isinstance(field, dict):
                    f_id = field.get('@id', None)
                    f_name = field.get('name', '<no name>')
                    print(f"    - {f_id} ({f_name})")
                else:
                    print(f"    - {field}")
        elif 'column' in rs:
            print("  Columns:")
            for col in rs['column']:
                c_id = col.get('@id', col)
                c_name = col.get('name', '<no name>')
                print(f"    - {c_id} ({c_name})")
        else:
            print("  No fields defined.")
        print()

## 3. Data Extraction
Load data from the key record set(s) into a DataFrame. All extraction is performed by referencing the record set and field `@id`s as discovered above.

In [ ]:
# Prepare to extract data from available record sets (by @id)

available_record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Record set IDs detected:", available_record_set_ids)
dataframes = {}

for record_set_id in available_record_set_ids:
    print(f"\nExtracting records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame shape for '{record_set_id}':", df.shape)
    if not df.empty:
        # Display a preview
        print("Columns:", df.columns.tolist())
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply simple filtering and normalization to a numeric field in the main record set for demonstration.

Replace the following IDs according to the previously printed field names (e.g., a protein abundance or MW field, referenced by its `@id` or column name).

In [ ]:
# Choose a main record set and a numeric field (by @id or column name)
# Example (you may need to adjust these IDs based on the overview above):

# For demonstration, let's auto-detect a numeric field
main_rs_id = available_record_set_ids[0] if available_record_set_ids else None
df = dataframes.get(main_rs_id)

if df is not None and not df.empty:
    # Try to detect numeric column
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_cols:
        # Try to convert columns to number where possible (since mass-spec data may use strings for numbers)
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.3f} ({len(filtered_df)} records):")
        display(filtered_df.head())
        
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to group by a categorical field (e.g., sample or protein ID)
        group_fields = [c for c in df.columns if c != numeric_field and df[c].dtype == 'object']
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped means by '{group_field}':")
            display(grouped_df.head())
    else:
        print("No numeric fields detected in this record set.")
else:
    print("No data extracted for the main record set.")

## 5. Visualization
Visualize data distributions or relationships between fields (e.g., histogram of a numeric field, scatter plot of two numeric fields).

Below is an example using matplotlib to plot a histogram and scatter plot.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if df is not None and not df.empty and numeric_cols:
    # Histogram for the main numeric field
    plt.figure(figsize=(7, 4))
    df[numeric_field].dropna().hist(bins=30, color='skyblue')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.title(f"Histogram of {numeric_field}")
    plt.show()

    # If there is a second numeric field, show scatter
    if len(numeric_cols) > 1:
        second_field = numeric_cols[1]
        plt.figure(figsize=(6,5))
        plt.scatter(df[numeric_field], df[second_field], alpha=0.6)
        plt.xlabel(numeric_field)
        plt.ylabel(second_field)
        plt.title(f"Scatter Plot of {numeric_field} vs {second_field}")
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to explore the structure and tabular data of the Mass Spectrometry Human Mast Cell dataset using only entity `@id` references. You can further extend this notebook for in-depth proteomics or statistical analysis based on your needs.